In [1]:
from selenium.common import NoSuchElementException

from functions import configure_chrome, login, read_group, save_raw_posts
from setup import FACEBOOK_USERNAME, FACEBOOK_PASSWORD, facebook_groups

In [2]:
driver, wait = configure_chrome()
login(driver=driver,
      wait=wait,
      username=FACEBOOK_USERNAME,
      password=FACEBOOK_PASSWORD)



Please solve the CAPCHA


In [11]:
raw_posts = []

for group_info in facebook_groups:
    group_posts = read_group(driver=driver,
                             group_info=group_info)
    raw_posts += group_posts


In [11]:
save_raw_posts(raw_posts=raw_posts)

In [8]:
driver.close()

In [3]:
"""Functions for scraping data from Facebook's groups."""
import os
import random
import time
from datetime import datetime
import yaml

import pandas as pd

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.support.wait import WebDriverWait
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC

from scraper.models import PostModel, GroupModel


In [11]:
driver.get("https://www.facebook.com/groups/366245597210651")

In [ ]:
# Start from bottom, untill fits last seen

In [167]:
post_elements = driver.find_elements(By.CSS_SELECTOR, "div[role='feed'] > div")


post_elements[8].find_element(By.CSS_SELECTOR, "[title*='Shared with']")

<selenium.webdriver.remote.webelement.WebElement (session="94f05d4667fa51245562073773716df9", element="f.6B9000857D6B8F72C92D70B6B13F574A.d.CB2C49F31AF65E11503053C5DEBB1A79.e.11858")>

In [212]:
# Find last
from functions import to_datetime, rest, scroll


def last_idx(post_elements):
    n = len(post_elements)
    for i, post_element in enumerate(reversed(post_elements)):
        try:
            globe = post_element.find_element(By.CSS_SELECTOR, "[title*='Shared with']")
            return n - i - 1
        except:
            continue
    return None



def get_created_at(post_element):
    globe = post_element.find_element(By.CSS_SELECTOR, "[title*='Shared with']")
    panel = globe.find_element(By.XPATH, "../../../..")
    share = panel.find_elements(By.XPATH, "./*")[-3]

    ActionChains(driver).move_to_element(share).perform()
    rest()

    created_at = driver.find_element(By.CSS_SELECTOR, "[role='tooltip']").text
    return to_datetime(created_at)


In [222]:
last_update = datetime(2026, 4, 29)

In [ ]:
# Scroll till reach last_update
# Find all posts elements
# From Last parse

def scroll_till_last(driver):
    while True:
        idx = last_idx(post_elements)

        last_post_created_at = get_created_at(post_elements[idx])

        if last_post_created_at > last_update:
            scroll(driver=driver)